# Task 02 — Silver: Orders Clean & Enrich

**Workshop: Final Pipeline | Layer 2 of 3**

## Goal

Read `bronze.lab_orders`, apply five transformations, write to `silver.lab_orders`.

```
bronze.lab_orders
        |
        v  cast * withColumn * when/otherwise * filter * drop
        v
silver.lab_orders
```

## Transformations to implement

| # | What | Details |
|---|------|---------|
| 1 | `cast` | `order_datetime` String -> TimestampType |
| 2 | `withColumn` | Add `net_amount` = `total_amount` x (1 - `discount_percent` / 100), rounded to 2 dp |
| 3 | `when / otherwise` | Add `order_tier`: `"Premium"` >= 500 * `"Standard"` >= 200 * `"Small"` otherwise |
| 4 | `filter` | Remove rows where `order_id` IS NULL |
| 5 | `drop` | Remove `_source_file` (Bronze metadata, not needed in Silver) |

> `net_amount` is used by Task 03 for revenue — make sure the formula is correct.


In [ ]:
%run ../../setup/00_setup

In [ ]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders"
target_table = f"{catalog}.{silver_schema}.lab_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import all PySpark functions needed for the five transformations above.


In [ ]:
# HINT -- pyspark.sql.functions:
#
#   from pyspark.sql.functions import col, when, round, to_timestamp, lit
#   from pyspark.sql.types import TimestampType
#
#   col("name")
#       -> reference to a DataFrame column by name
#
#   to_timestamp(col_expr, format_string)
#       -> parse String -> TimestampType
#          Java SimpleDateFormat tokens:
#            yyyy  year    MM  month   dd  day
#            HH    hour    mm  minute  ss  second
#            'T'   literal character (must be quoted in the pattern)
#          This dataset uses: "yyyy-MM-dd'T'HH:mm:ss"
#
#   when(condition, value).when(...).otherwise(value)
#       -> CASE WHEN ... THEN ... ELSE ... END
#          Always end with .otherwise() to cover remaining rows.
#
#   round(column_expression, scale)
#       -> round a Column expression to N decimal places
#
#   lit(python_value)
#       -> wrap a Python literal as a Column expression
#          Required when mixing Python int/float with col() in arithmetic.
#          e.g. col("a") * (lit(1) - col("b") / lit(100))

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 2 — Read the Bronze table


In [ ]:
# HINT -- spark.table():
#
#   df = spark.table("catalog.schema.table_name")
#
#   Reads a managed Delta table from Unity Catalog.
#   Equivalent to: spark.sql("SELECT * FROM catalog.schema.table_name")
#   No file path needed.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
print(f"Bronze rows : {orders_bronze.count():,}")
orders_bronze.printSchema()

---

## Exercise 3 — Apply all five transformations in one chain

Chain all transformations into a single expression. Name the result `orders_silver`.


In [ ]:
# HINT -- chaining transformations:
#
#   orders_silver = (
#       df
#       # 1. replace order_datetime with Timestamp
#       .withColumn("order_datetime",
#           to_timestamp(col("order_datetime"), "format_pattern"))
#
#       # 2. add net_amount: total_amount * (1 - discount_percent / 100)
#       .withColumn("net_amount",
#           round(col("col_a") * (lit(1) - col("col_b") / lit(100)), 2))
#
#       # 3. add order_tier using when/otherwise
#       .withColumn("order_tier",
#           when(col("amount_col") >= threshold1, "Label1")
#           .when(col("amount_col") >= threshold2, "Label2")
#           .otherwise("Label3"))
#
#       # 4. keep only rows where key_col is not null
#       .filter(col("key_col").isNotNull())
#
#       # 5. remove a column by name
#       .drop("column_name")
#   )
#
#   Timestamp format for this dataset: "yyyy-MM-dd'T'HH:mm:ss"

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.select(
    "order_id","order_datetime","total_amount",
    "discount_percent","net_amount","order_tier"
).show(10, truncate=False)
orders_silver.printSchema()

---

## Exercise 4 — Write to Silver as a managed Delta table


In [ ]:
# HINT -- write with overwriteSchema:
#
#   df.write
#       .format("delta")
#       .mode("overwrite")
#       .option("overwriteSchema", "true")
#       .saveAsTable("catalog.schema.table")
#
#   overwriteSchema=true is needed when the new DataFrame has columns
#   that did not exist in the previous table version.
#   Without it, Spark raises AnalysisException on schema mismatch.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} rows")
display(spark.table(target_table).limit(5))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))